# **Notebook 4: Sentiment Analysis**


> **Objective:** Perform initial data profiling on the raw data to justify our preprocessing decisions and to uncover actionable business insights and guide Machine Learning feature selection.
The Exploratory Analysis section of this notebook aims at obtaining the most information about our data to better judge what techniques will be required regarding the possibility of missing values and outliers as well as what transformations will be made to the data to improve its usability.


In [7]:
import pandas as pd
import numpy as np
import re
import pickle
!pip install vaderSentiment

In [2]:
with open('train_dataset_preprocessed.pkl', 'rb') as f:
    train_dataset = pickle.load(f)

In [5]:
train_dataset['lyrics_regex_cleaned_for_sentiment']

0         When you said you loved me Did you mean it the...
1         Kid Naruto Rap Everybody wants you to hurt Eve...
2         People tell me I've changed I find it hard to ...
3         You've been warned about me Don't try to get t...
4         Oh yeah Oh yeah yeah yeah Oh yeah Right there ...
                                ...                        
133029    If I knew when I was young that I'd be older T...
133030    Cash Wah-wah-wah Wah-wah-wah Everything is chr...
133031    I was born inside a home The youngest of the t...
133032    Rollin' by the pool the falling stars are not ...
133033    Yeah Epic Games, I ain't playin' no games Now,...
Name: lyrics_regex_cleaned_for_sentiment, Length: 133034, dtype: object

NameError: name 'vader_wrapper' is not defined

In [9]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [10]:
def vader_wrapper(user_review):
    """
    Returns a dict with keys: 'compound', 'pos', 'neg', 'neu'.
    If user_review is a list, returns the mean of each score across sentences.
    If user_review is a single string, returns VADER's scores for that string.
    """
    vader = SentimentIntensityAnalyzer()

    if isinstance(user_review, list):
        pos_list, neg_list, neu_list, comp_list = [], [], [], []
        for sentence in user_review:
            scores = vader.polarity_scores(sentence)
            pos_list.append(scores["pos"])
            neg_list.append(scores["neg"])
            neu_list.append(scores["neu"])
            comp_list.append(scores["compound"])
        return {
            "compound": float(np.mean(comp_list)),
            "pos": float(np.mean(pos_list)),
            "neg": float(np.mean(neg_list)),
            "neu": float(np.mean(neu_list)),
        }
    else:
        scores = vader.polarity_scores(user_review)
        return {k: float(scores[k]) for k in ("compound", "pos", "neg", "neu")}


In [11]:
vader_scores = train_dataset["lyrics_regex_cleaned_for_sentiment"].map(vader_wrapper).apply(pd.Series)
dataset = pd.concat([train_dataset, vader_scores.add_prefix("content_polarity_vader_")], axis=1)

In [17]:
print("\nReview:\n")
print(train_dataset[["lyrics_regex_cleaned_for_sentiment"]]
      .sample(2, random_state=42).values)

print('\nVader Compound:\n', dataset[['content_polarity_vader_compound']].sample(2, random_state=42).values)
print('\nVader Positive:\n', dataset[['content_polarity_vader_pos']].sample(2, random_state=42).values)
print('\nVader Negative:\n', dataset[['content_polarity_vader_neg']].sample(2, random_state=42).values)
print('\nVader Neutral:\n', dataset[['content_polarity_vader_neu']].sample(2, random_state=42).values)


Review:

[['(I pledge allegiance to Budweiser and free drugs) Peace to Rick James, Anna Nicole Smith, Bill Clinton And Motley Crue And anyone else who has ever utilized their 15 minutes Of fame to realize their true dreams of being an Absolute jerk off, just to keep the masses entertained This goes out to learning from the mistakes of others Bring it on now Come on I said come on I said come on They call me a jerk, once they get to know me But they don\'t stop calling, they read me well It\'s no work if I was phony, I\'d win a trophy Who needs to make records when there\'s seeds to sell Freak the bell, and make it all spin crooked God please help, too much grim to look at Grab the tree by the limb and shook it Like, "Have you seen my self esteem, where the hell\'d you put it?" Oh wait, never mind, I found it in a bottle Drunk at the Troubadour talking to a model Wrecked the rental on Santa Monica Boulevard I was headed to the El Rey to slap a security guard Rowdy, stubborn, loud and a